# 01 — Data Collection & Alignment\n\n**BBL514E Pattern Recognition — Term Project**\n\nBu notebook:\n1. BTC ve ETH fiyat verilerini indirir (yfinance)\n2. Makroekonomik verileri indirir (S&P 500, Gold, DXY, VIX, Treasury Yields)\n3. Tüm verileri kripto günlük timeline'a hizalar\n4. Temel veri kalitesi kontrollerini yapar\n\n**Çıktılar:**\n- `data/raw/` → Ham veriler\n- `data/processed/` → Hizalanmış veriler (btc_aligned.csv, eth_aligned.csv)

In [ ]:
import sys\nsys.path.insert(0, \"..\")\n\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport matplotlib.dates as mdates\nimport seaborn as sns\n\nplt.style.use(\"seaborn-v0_8-darkgrid\")\npd.set_option(\"display.max_columns\", 50)\npd.set_option(\"display.float_format\", lambda x: f\"{x:,.2f}\")

## 1. Fiyat Verisi Toplama (BTC, ETH)

In [ ]:
from src.data.price_collector import collect_all_prices\n\nprices = collect_all_prices(save=True)\n\nfor coin, df in prices.items():\n    print(f\"\\n{'='*50}\")\n    print(f\"{coin.upper()} — {len(df)} rows\")\n    print(f\"Date range: {df.index.min().date()} to {df.index.max().date()}\")\n    print(f\"NaN count: {df.isna().sum().sum()}\")\n    print(df.describe())

In [ ]:
# Fiyat grafikleri\nfig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)\n\nfor ax, (coin, df) in zip(axes, prices.items()):\n    ax.plot(df.index, df[\"Close\"], linewidth=0.8)\n    ax.set_title(f\"{coin.upper()}/USD — Daily Close Price\", fontsize=13)\n    ax.set_ylabel(\"Price (USD)\")\n    ax.xaxis.set_major_locator(mdates.YearLocator())\n    ax.xaxis.set_major_formatter(mdates.DateFormatter(\"%Y\"))\n    ax.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.savefig(\"../reports/price_history.png\", dpi=150, bbox_inches=\"tight\")\nplt.show()

## 2. Makroekonomik Veri Toplama

In [ ]:
from src.data.macro_collector import collect_all_macro\n\nmacro = collect_all_macro(save=True)\n\nprint(\"\\n=== Daily Macro ===\")\nprint(macro[\"daily\"].describe())\n\nprint(\"\\n=== Yield Proxies ===\")\nprint(macro[\"yields\"].describe())\n\nif macro.get(\"fred\") is not None:\n    print(\"\\n=== FRED Monthly ===\")\n    print(macro[\"fred\"].describe())\nelse:\n    print(\"\\nFRED data not available (no API key). Using yield proxies as rate-environment data.\")

In [ ]:
# Makro göstergeler zaman serisi\nfig, axes = plt.subplots(4, 2, figsize=(16, 14))\n\n# Daily macro\ndaily = macro[\"daily\"]\nfor ax, col in zip(axes[:, 0], daily.columns):\n    ax.plot(daily.index, daily[col], linewidth=0.7, color=\"steelblue\")\n    ax.set_title(col, fontsize=11)\n    ax.grid(True, alpha=0.3)\n\n# Yield proxies\nyields = macro[\"yields\"]\nfor ax, col in zip(axes[:, 1], yields.columns):\n    ax.plot(yields.index, yields[col], linewidth=0.7, color=\"darkorange\")\n    ax.set_title(col, fontsize=11)\n    ax.grid(True, alpha=0.3)\n\nplt.suptitle(\"Macroeconomic Indicators (2021-2025)\", fontsize=14, y=1.01)\nplt.tight_layout()\nplt.savefig(\"../reports/macro_indicators.png\", dpi=150, bbox_inches=\"tight\")\nplt.show()

## 3. Veri Hizalama (Alignment)

In [ ]:
from src.data.data_aligner import create_aligned_dataset\n\naligned = {}\nfor coin in [\"btc\", \"eth\"]:\n    aligned[coin] = create_aligned_dataset(\n        price_df=prices[coin],\n        macro_daily=macro[\"daily\"],\n        macro_yields=macro[\"yields\"],\n        fred_monthly=macro.get(\"fred\"),\n        coin_name=coin,\n        save=True,\n    )\n    print(f\"\\n{coin.upper()} aligned: {aligned[coin].shape}\")\n    print(f\"  Date range: {aligned[coin].index.min().date()} to {aligned[coin].index.max().date()}\")\n    print(f\"  Columns: {list(aligned[coin].columns)}\")

## 4. Veri Kalitesi Kontrolleri

In [ ]:
# NaN kontrolu\nfor coin in [\"btc\", \"eth\"]:\n    df = aligned[coin]\n    print(f\"\\n{coin.upper()} — NaN check after alignment:\")\n    nan_count = df.isna().sum()\n    if nan_count.sum() == 0:\n        print(\"  No NaN values.\")\n    else:\n        print(nan_count[nan_count > 0])\n\n# Duplicate index kontrolu\nfor coin in [\"btc\", \"eth\"]:\n    df = aligned[coin]\n    dupes = df.index.duplicated().sum()\n    print(f\"{coin.upper()} — Duplicate dates: {dupes}\")\n\n# Monotonic index kontrolu\nfor coin in [\"btc\", \"eth\"]:\n    df = aligned[coin]\n    print(f\"{coin.upper()} — Index is monotonic increasing: {df.index.is_monotonic_increasing}\")

In [ ]:
# Korelasyon matrisi — hizalanmış veri\nfig, axes = plt.subplots(1, 2, figsize=(16, 6))\n\nfor ax, coin in zip(axes, [\"btc\", \"eth\"]):\n    corr = aligned[coin].corr()\n    mask = np.triu(np.ones_like(corr, dtype=bool))\n    sns.heatmap(corr, mask=mask, annot=True, fmt=\".2f\", cmap=\"RdBu_r\",\n                center=0, vmin=-1, vmax=1, ax=ax, annot_kws={\"size\": 7})\n    ax.set_title(f\"{coin.upper()} — Correlation Matrix\", fontsize=12)\n\nplt.tight_layout()\nplt.savefig(\"../reports/correlation_matrix.png\", dpi=150, bbox_inches=\"tight\")\nplt.show()

## 5. Rapor İçin Veri Özet Tablosu

In [ ]:
# Final raporda kullanılacak veri özet tablosu\nsummary_rows = []\nfor coin in [\"btc\", \"eth\"]:\n    df = aligned[coin]\n    summary_rows.append({\n        \"Asset\": coin.upper(),\n        \"Source\": \"yfinance\",\n        \"Period\": f\"{df.index.min().date()} to {df.index.max().date()}\",\n        \"Total Rows\": len(df),\n        \"OHLCV Columns\": 5,\n        \"Macro Columns\": df.shape[1] - 5,\n        \"Total Columns\": df.shape[1],\n        \"NaN After Align\": df.isna().sum().sum(),\n    })\n\nsummary = pd.DataFrame(summary_rows)\nprint(\"\\n=== DATASET SUMMARY (for final report) ===\")\nprint(summary.to_string(index=False))

## Checkpoint\n\n**Senin incelemen gereken noktalar:**\n\n1. Fiyat grafikleri mantıklı mı? Bilinen bull/bear dönemleri görünüyor mu?\n2. Makro göstergeler beklediğin gibi mi? (VIX spike'ları, yield curve inversion, vs.)\n3. Korelasyon matrisi sürpriz bir şey gösteriyor mu?\n4. 1817 satır yeterli mi yoksa daha farklı bir tarih aralığı deneyelim mi?\n5. Yield proxy'ler (10Y, 5Y, 3M) FRED verisi yerine yeterli mi, yoksa FRED API key bulalım mı?\n\n**Bu notebook'u PyCharm'da çalıştır, grafikleri incele, sonra bana dönüş yap.**